# Lesson 3.2: The Binomial Distribution

Use the binomial distribution for the lesson's drug that works for 70% of patients (10 patients): exactly, at most and at least, plus mean, SD and a simulation check.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. Exactly 7 respond

binom.pmf(k, n, p) gives P(X = k), one bar of the distribution. We also compute the lesson's formula C(n, k) x p^k x (1 - p)^(n - k) by hand to see that they agree.

In [1]:
import math
import numpy as np
from scipy.stats import binom

n, p = 10, 0.7

by_hand = math.comb(n, 7) * p**7 * (1 - p)**3
print(f'formula   : {by_hand:.4f}')
print(f'binom.pmf : {binom.pmf(7, n, p):.4f}')

formula   : 0.2668
binom.pmf : 0.2668


## 2. At most, more than, at least

cdf(k) is P(X <= k), 'at most k'. sf(k) is the survival function P(X > k), 'more than k'. So 'at least 9' is sf(8), which is the same as 1 minus cdf(8).

In [2]:
print(f'P(X <= 6)  at most 6      = {binom.cdf(6, n, p):.4f}')
print(f'P(X > 8)   more than 8      = {binom.sf(8, n, p):.4f}')
print(f'P(X >= 9)  at least 9       = {1 - binom.cdf(8, n, p):.4f}')
print(f'P(9) + P(10) added by hand  = {binom.pmf(9, n, p) + binom.pmf(10, n, p):.4f}')

P(X <= 6)  at most 6      = 0.3504
P(X > 8)   more than 8      = 0.1493
P(X >= 9)  at least 9       = 0.1493
P(9) + P(10) added by hand  = 0.1493


## 3. Mean and standard deviation

The shortcuts are mean = n x p and SD = square root of n x p x (1 - p). SciPy's stats method returns the same values from the distribution itself.

In [3]:
print(f'mean = n*p = {n * p:.1f}')
print(f'variance   = {n * p * (1 - p):.2f}')
print(f'SD         = {math.sqrt(n * p * (1 - p)):.3f}')

m, v = binom.stats(n, p)
print(f'scipy      : mean {float(m):.1f}, variance {float(v):.2f}')

mean = n*p = 7.0
variance   = 2.10
SD         = 1.449
scipy      : mean 7.0, variance 2.10


## 4. Check by simulation

Run the 10-patient trial 100,000 times with a seeded generator and count how many responders each trial has. The simulated shares should be close to the exact probabilities.

In [4]:
rng = np.random.default_rng(42)
responders = rng.binomial(n, p, size=100_000)

print(f'simulated P(X = 7)  = {(responders == 7).mean():.4f}  (exact {binom.pmf(7, n, p):.4f})')
print(f'simulated P(X >= 9) = {(responders >= 9).mean():.4f}  (exact {binom.sf(8, n, p):.4f})')
print(f'simulated mean {responders.mean():.3f}, SD {responders.std():.3f}')

simulated P(X = 7)  = 0.2679  (exact 0.2668)
simulated P(X >= 9) = 0.1485  (exact 0.1493)
simulated mean 6.997, SD 1.450


## Try this

- Change p to 0.5 and predict where the most likely count moves, then print binom.pmf for k = 0 to 10.
- A coin is flipped 100 times and gives 62 heads. Compute binom.sf(61, 100, 0.5) and compare with the lesson's 0.0105.

## The same idea in R, SPSS and Stata

**R**

```r
n <- 10; p <- 0.7
dbinom(7, n, p)                       # exactly 7
pbinom(6, n, p)                       # at most 6
pbinom(8, n, p, lower.tail = FALSE)   # more than 8 = at least 9
n * p
sqrt(n * p * (1 - p))

set.seed(42)
x <- rbinom(100000, n, p)
mean(x == 7)
```

**SPSS** (Transform > Compute Variable (Function group: PDF & Noncentral PDF, CDF & Noncentral CDF, Random Numbers))

```
DATA LIST FREE / dummy.
BEGIN DATA
1
END DATA.
COMPUTE exact7 = PDF.BINOM(7, 10, 0.7).
COMPUTE atmost6 = CDF.BINOM(6, 10, 0.7).
COMPUTE atleast9 = 1 - CDF.BINOM(8, 10, 0.7).
COMPUTE mean_x = 10 * 0.7.
COMPUTE sd_x = SQRT(10 * 0.7 * 0.3).
FORMATS exact7 atmost6 atleast9 sd_x (F8.4).
LIST VARIABLES=exact7 atmost6 atleast9 mean_x sd_x.
```

**Stata**

```
display binomialp(10, 7, 0.7)
display binomial(10, 6, 0.7)
display binomialtail(10, 9, 0.7)
display 10 * 0.7
display sqrt(10 * 0.7 * 0.3)

clear
set seed 42
set obs 100000
gen x = rbinomial(10, 0.7)
gen is7 = (x == 7)
summarize is7
```